# 01 - Data Ingestion & Exploration
## InsightForge Agentic RAG Pipeline

This notebook demonstrates the data ingestion pipeline:
1. CSV upload & profiling
2. ID column detection & removal
3. Schema analysis
4. Embedding generation for RAG
5. FAISS vector index creation

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from pathlib import Path
from ingestion.csv_profiler import CSVProfiler
from config import UPLOAD_FOLDER

print('Modules loaded successfully!')

## Step 1: Load & Profile a CSV Dataset

The `CSVProfiler` analyzes the dataset structure, detects column types,
identifies ID/junk columns, and generates metadata for the Planner agent.

In [ ]:
# Create sample dataset for demonstration
np.random.seed(42)
n = 500
sample_df = pd.DataFrame({
    'customer_id': range(1, n+1),
    'order_id': [f'ORD-{i:05d}' for i in range(1, n+1)],
    'product_category': np.random.choice(['Electronics', 'Clothing', 'Books', 'Home', 'Sports'], n),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'revenue': np.random.uniform(10, 500, n).round(2),
    'quantity': np.random.randint(1, 20, n),
    'rating': np.random.uniform(1, 5, n).round(1),
})

# Save to disk
sample_path = '../data/sample_sales.csv'
sample_df.to_csv(sample_path, index=False)
print(f'Sample dataset: {sample_df.shape[0]} rows x {sample_df.shape[1]} columns')
sample_df.head()

In [ ]:
# Profile the CSV
profiler = CSVProfiler()
profile = profiler.profile(sample_path)

print(f'Filename: {profile["filename"]}')
print(f'Row count: {profile["row_count"]}')
print(f'\nColumn Analysis:')
print('-' * 70)
for col_name, col_info in profile['columns'].items():
    dtype = col_info['dtype']
    unique = col_info['unique_count']
    is_id = col_info.get('is_id', False)
    flag = ' [ID - WILL BE DROPPED]' if is_id else ''
    print(f'  {col_name:20s} | {dtype:8s} | unique={unique:5d} | numeric={col_info["is_numeric"]}{flag}')

## Step 2: ID Column Detection

The profiler automatically detects columns that are identifiers based on:
1. **Name matching**: Exact match against known ID patterns (id, uuid, order_id)
2. **Suffix + cardinality**: Suffix like `_id` with >80% unique values
3. **Sequential detection**: Near-unique numeric columns with step=1 diffs

In [ ]:
# Show which columns will be dropped
dropped = profile.get('dropped_columns', [])
print(f'Columns flagged for removal: {dropped}')
print(f'These columns have near-100% uniqueness and are not useful for analysis.')

# Clean the dataframe
clean_df = profiler.clean_dataframe(sample_df.copy(), profile)
print(f'\nBefore cleaning: {sample_df.shape[1]} columns')
print(f'After cleaning:  {clean_df.shape[1]} columns')
print(f'Remaining columns: {list(clean_df.columns)}')

## Step 3: Document Ingestion for RAG

The `DocumentIngestor` takes text, splits it into chunks (sentences),
embeds each chunk using SentenceTransformers (all-MiniLM-L6-v2, 384-dim),
and indexes them in a FAISS flat inner-product index for fast retrieval.

In [ ]:
from ingestion.document_ingestor import DocumentIngestor

# Create text summary of the dataset for RAG
cats = ', '.join(clean_df['product_category'].unique())
regions = ', '.join(clean_df['region'].unique())
summary_text = (
    f'Sales dataset with {len(clean_df)} records. '
    f'Product categories include {cats}. '
    f'Regions covered: {regions}. '
    f'Average revenue is ${clean_df["revenue"].mean():.2f} per transaction. '
    f'Average quantity sold is {clean_df["quantity"].mean():.1f} items. '
    f'Average customer rating is {clean_df["rating"].mean():.2f} out of 5. '
    f'Revenue ranges from ${clean_df["revenue"].min():.2f} to ${clean_df["revenue"].max():.2f}. '
    f'Total revenue: ${clean_df["revenue"].sum():,.2f}.'
)

print('Dataset summary for RAG ingestion:')
print(summary_text)

In [ ]:
# Ingest into vector store
ingestor = DocumentIngestor()
ingestor.ingest(summary_text, source='sample_sales.csv')

print(f'Chunks indexed: {len(ingestor.chunks)}')
print(f'FAISS index size: {ingestor.index.ntotal}')
print(f'Embedding dimension: 384')
print(f'\nSample chunks:')
for i, chunk in enumerate(ingestor.chunks[:3]):
    print(f'  [{i}] {chunk[:80]}...')

---
## Summary

This notebook demonstrated:
- CSV profiling with automatic schema detection
- ID column auto-removal (customer_id, order_id detected and dropped)
- Text summarization of dataset for RAG
- SentenceTransformer embedding + FAISS indexing

**Next**: See `02_agentic_rag_pipeline.ipynb` for the Planner -> Executor -> Synthesizer pipeline.